# Portfolio Benchmark

This notebook imports saved mean-variance portfolio runs from `results/portfolio`. It is analysis-only and does not run training.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from mfc.visualization import (
    best_runs_by_label,
    gradient_diagnostics,
    load_runs,
    objective_table,
    plot_state_flow,
    plot_validation_rewards,
    runtime_table,
    transport_correction_table,
)

ENV = 'portfolio'
RESULTS_ROOT = ROOT / 'results'
runs = load_runs(RESULTS_ROOT, env=ENV)
print(f'Loaded {len(runs)} saved runs from {RESULTS_ROOT / ENV}')

## Validation Reward

Mean validation reward over training, with one standard deviation across seeds. The terminal reward is expected wealth penalized by terminal variance.

In [ ]:
if not runs:
    print('No saved runs yet. Run scripts/run.py before executing the analysis cells.')
else:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    plot_validation_rewards(runs, env=ENV, horizon=10, flow='exact', ax=ax)
    ax.set_title('Portfolio validation reward')
    plt.show()

## Wealth Mean and Variance Flow

Population wealth mean and variance induced by the best saved policy for each algorithm and perturbation level.

In [ ]:
for run in best_runs_by_label(runs):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    plot_state_flow(run, ax=ax)
    meta = run['metadata']
    ax.set_title(f"{meta['algorithm']}, lambda={meta['perturbation']}")
    plt.show()

## Objective Table: J vs J^lambda

Closed-form objective values at the learned policy, with the analytical unperturbed optimum included when available.

In [ ]:
display(objective_table(runs))

## Transport Gradient Diagnostics

Optional Monte Carlo check of bias, MSE, cosine similarity, and perturbation bias against the exact portfolio gradient oracle.

In [ ]:
RUN_GRADIENT_DIAGNOSTICS = False
N_REPLICATIONS = 20

if RUN_GRADIENT_DIAGNOSTICS:
    tables = []
    for run in runs:
        try:
            tables.append(gradient_diagnostics(run, n_replications=N_REPLICATIONS))
        except ValueError:
            pass
    display(pd.concat(tables, ignore_index=True) if tables else pd.DataFrame())
else:
    print('Set RUN_GRADIENT_DIAGNOSTICS = True to recompute gradient bias/MSE diagnostics.')

## Missing Mean-Field Correction Term

Optional diagnostic comparing the full transport gradient with the policy-only term. The difference is the mean-field correction absent from classical REINFORCE.

In [ ]:
RUN_CORRECTION_DIAGNOSTIC = False

if RUN_CORRECTION_DIAGNOSTIC:
    tables = []
    for run in runs:
        try:
            tables.append(transport_correction_table(run, n_replications=N_REPLICATIONS))
        except ValueError:
            pass
    display(pd.concat(tables, ignore_index=True) if tables else pd.DataFrame())
else:
    print('Set RUN_CORRECTION_DIAGNOSTIC = True to estimate the missing correction term.')

## Runtime Table

Average elapsed time and estimated simulator budget for each algorithm configuration.

In [ ]:
display(runtime_table(runs))